# Data Science in Cyber — Final Project

**Reproduction and critical evaluation of a network intrusion detection tutorial**

This notebook reproduces and critically evaluates the methodology from:

> *AI-Powered Cyber Security Part 2: Building an Intrusion Detection System with Python and Scikit-learn* — Chandan Bhagat

All metrics below were generated by running this notebook on the NSL-KDD dataset.


## 1. Introduction and Source Summary

### Problem
Network intrusion detection (NIDS) identifies malicious connection patterns in network traffic. Signature-based tools miss novel attacks; machine learning can learn normal vs. attack behavior from connection-level features.

### Why it matters
Modern attacks often evade static signatures. Missing an attack (false negative) can allow compromise; excessive alerts (false positives) overwhelm security teams.

### Selected source
- **Article:** AI-Powered Cyber Security Part 2: Building an Intrusion Detection System with Python and Scikit-learn
- **Link:** https://chandanbhagat.com.np/ai-powered-cyber-security-part-2-building-an-intru/
- **Reference GitHub (similar IDS project):** https://github.com/akinyeraakintunde/network-intrusion-detection-ml
- **Note:** The blog author publishes full code inline; no dedicated repository is linked in the article.

### Dataset
**NSL-KDD** — 41 connection features + attack label + difficulty score.
- Official page: https://www.unb.ca/cic/datasets/nsl.html
- Files used: `KDDTrain+.txt`, `KDDTest+.txt`

### Proposed solution
1. Encode categorical features and scale numeric features
2. Handle class imbalance with SMOTE on the training split only
3. Train **Random Forest** for supervised attack detection
4. Train **Isolation Forest** on normal traffic for anomaly scoring

### Author's main claims
1. Random Forest achieves ~99% precision/recall on a held-out split of the training file
2. ROC-AUC ≈ 0.9987
3. Training finishes in under 5 minutes on a laptop CPU
4. Isolation Forest complements RF for unknown threats
5. Real-world performance will be lower than benchmark numbers


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loading import load_train_test, get_feature_columns
from src.eda import (
    save_data_overview,
    plot_class_distribution,
    plot_missing_values,
    plot_numeric_distributions,
    plot_outlier_boxplots,
    plot_service_attack_crosstab,
    plot_correlation_heatmap,
)
from src.preprocessing import detect_redundant_features, build_preprocessor, apply_smote
from src.models import train_models
from src.evaluation import evaluate_model_bundle, build_comparison_table, compute_classification_metrics
from src.error_analysis import extract_error_examples, summarize_error_patterns
from src.utils import ensure_output_dirs, TABLES_DIR, FIGURES_DIR, METRICS_DIR

ensure_output_dirs()
print("Environment ready. Outputs will be saved under results/")


## 2. Data Loading and Inspection


In [ ]:
train_df, test_df = load_train_test()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nFirst rows (train):")
display(train_df.head())
print("\nColumn names:")
print(list(train_df.columns))
print("\nIndex:", train_df.index[:5].tolist(), "— default RangeIndex is expected for NSL-KDD TXT files")
print("\nData types:")
print(train_df.dtypes)
print("\nTarget columns:")
print("- label: original attack name (e.g., normal, neptune)")
print("- binary_label: 0=normal, 1=attack (used for modeling)")
print("- difficulty: NSL-KDD difficulty score (not used as a feature)")

overview = pd.DataFrame({
    "missing_values": train_df.isna().sum(),
    "duplicate_rows": [train_df.duplicated().sum()] + [None]*(len(train_df.columns)-1),
})
print("\nMissing values (train):", int(train_df.isna().sum().sum()))
print("Duplicate rows (train):", int(train_df.duplicated().sum()))

constant_cols = [c for c in train_df.columns if train_df[c].nunique(dropna=False) <= 1]
print("Single-value columns:", constant_cols)

duplicated_col_names = train_df.columns[train_df.T.duplicated(keep=False)].tolist()
print("Duplicated columns:", duplicated_col_names)
print("Irrelevant columns for modeling: label, difficulty, attack_category")

save_data_overview(train_df, prefix="train")
save_data_overview(test_df, prefix="test")

class_dist = pd.read_csv(TABLES_DIR / "train_class_distribution.csv")
display(class_dist)

# NSL-KDD has no timestamp column; temporal analysis is not applicable.
print("No temporal columns are present in NSL-KDD.")


## 3. Exploratory Data Analysis

We inspect distributions, missing values, outliers, class imbalance, service/attack relationships, and correlations.

### Correlation method: Spearman
We use **Spearman rank correlation** because many numeric features are skewed, contain outliers, and have monotonic but non-linear relationships.

**Meaning:** measures monotonic association using ranked values.

**Assumptions:** ordinal/monotonic relationships; robust to outliers compared with Pearson.

**Limitations:** misses non-monotonic patterns; can understate strong linear links.

**Cybersecurity meaning:** high correlation among error-rate features suggests redundant signals from the same underlying scanning/flooding behavior — useful for feature redundancy checks, not causal inference.


In [ ]:
plot_class_distribution(train_df, "train")
plot_missing_values(train_df, "train")
plot_numeric_distributions(train_df, "train")
plot_outlier_boxplots(train_df, "train")
plot_service_attack_crosstab(train_df, "train")
plot_correlation_heatmap(train_df, "train")

print("Saved EDA figures to results/figures/")

# Practical interpretation
normal_pct = (train_df["binary_label"] == 0).mean()
print(f"Training normal traffic share: {normal_pct:.3f}")
print("The dataset is moderately imbalanced but not extreme.")
print("The original author addresses imbalance with SMOTE + class_weight='balanced'.")
print("NSL-KDD is a classic benchmark but dated; attack mix may not reflect current networks.")


## 4. Feature Engineering

Steps applied:
1. **One-hot encoding** for `protocol_type`, `service`, `flag` — converts categorical connection metadata to numeric model inputs.
2. **StandardScaler** on numeric features — prevents large byte-count columns from dominating distance/tree splits unfairly.
3. **SMOTE** on training data only — synthesizes minority-class examples to reduce attack under-representation during training.
4. **Redundancy detection** — constant and highly correlated columns are flagged; `num_outbound_cmds` is constant in this dataset.

All transformers are fit on training data only to avoid leakage.


In [ ]:
redundancy = detect_redundant_features(train_df)
print(json.dumps(redundancy, indent=2))

preprocessor = build_preprocessor()
x_train = train_df[get_feature_columns()]
y_train = train_df["binary_label"]
x_test = test_df[get_feature_columns()]
y_test = test_df["binary_label"]

x_train_enc = preprocessor.fit_transform(x_train)
x_test_enc = preprocessor.transform(x_test)
x_res, y_res = apply_smote(x_train_enc, y_train)

print("Encoded train shape:", x_train_enc.shape)
print("After SMOTE:", x_res.shape, "class balance:", pd.Series(y_res).value_counts().to_dict())


## 5. Model Training

We train:
- **Baseline:** Logistic Regression
- **Author-style model:** Random Forest (200 trees, max_depth=20, SMOTE)
- **Anomaly model:** Isolation Forest on normal traffic only

Evaluation uses:
1. **Official NSL-KDD test split** (`KDDTest+.txt`) — rigorous benchmark
2. **Author-style random 80/20 split on training file** — reproduces tutorial setup


In [ ]:
trained_official = train_models(train_df, test_df, use_official_test=True)
trained_author = train_models(train_df, test_df, use_official_test=False)

official = {
    "logistic_regression": evaluate_model_bundle(trained_official, "official_logistic_regression", trained_official.logistic_regression, trained_official.x_test_processed, trained_official.y_test.values),
    "random_forest": evaluate_model_bundle(trained_official, "official_random_forest", trained_official.random_forest, trained_official.x_test_processed, trained_official.y_test.values),
}

class IsoWrap:
    def __init__(self, model, x):
        self.pred = (model.predict(x) == -1).astype(int)
    def predict(self, x):
        return self.pred

official_iso = IsoWrap(trained_official.isolation_forest, trained_official.x_test_processed)
official["isolation_forest"] = evaluate_model_bundle(trained_official, "official_isolation_forest", official_iso, trained_official.x_test_processed, trained_official.y_test.values, use_proba=False)

comparison = build_comparison_table(official)
display(comparison)

author_rf_pred = trained_author.random_forest.predict(trained_author.x_test_processed)
author_rf_proba = trained_author.random_forest.predict_proba(trained_author.x_test_processed)[:, 1]
author_metrics = compute_classification_metrics(trained_author.y_test.values, author_rf_pred, author_rf_proba)
print("Author-style RF metrics:", json.dumps(author_metrics, indent=2))


## 6. Evaluation

### Metrics used

| Metric | Formula (binary) | Why it matters in IDS |
|--------|------------------|------------------------|
| Accuracy | (TP+TN)/N | Misleading when classes or costs are imbalanced |
| Precision | TP/(TP+FP) | Alert quality — false positives waste analyst time |
| Recall | TP/(TP+FN) | Attack capture — false negatives let intrusions through |
| F1 | 2PR/(P+R) | Balance between precision and recall |
| F2 | 5PR/(4P+R) | Weights recall higher — common in security |
| MCC | (TP·TN−FP·FN)/√(...) | Balanced measure even under imbalance |
| ROC-AUC | area under TPR vs FPR | Threshold-free ranking quality |

**False positives** = normal traffic flagged as attack → alert fatigue.

**False negatives** = attack missed → potential breach.

For intrusion detection, **recall on attacks (or F2)** should usually be prioritized over raw accuracy, while precision still matters for operational load.


In [ ]:
print("Official test comparison:")
display(comparison)

print("\nAccuracy can look acceptable while attack recall remains low.")
print("Example: Random Forest official-test recall for attacks =", round(official["random_forest"]["recall"], 4))


## 7. Error Analysis


In [ ]:
rf_pred = trained_official.random_forest.predict(trained_official.x_test_processed)
fp, fn = extract_error_examples(
    trained_official.x_test,
    trained_official.y_test,
    rf_pred,
    original_labels=test_df.loc[trained_official.x_test.index, "label"],
)
patterns = summarize_error_patterns(trained_official.x_test, trained_official.y_test, rf_pred)

print("False positive examples:")
display(fp.head())
print("False negative examples:")
display(fn.head())
print("Error patterns:")
display(patterns)

print("\nCybersecurity implications:")
print("- False positives on private/other UDP/TCP flows create noisy alerts.")
print("- False negatives on rare attack types (e.g., R2L/U2R) are especially dangerous.")
print("- For this problem, false negatives are usually more dangerous than false positives.")


## 8. Critical Evaluation of the Author's Claims


In [ ]:
claims = pd.DataFrame([
    {
        "Claim": "Random Forest reaches ~99% precision/recall on held-out data",
        "Author's Evidence": "classification_report in blog (~0.99 for both classes)",
        "Our Reproduction Evidence": f"Author-style split: acc={author_metrics['accuracy']:.4f}, recall={author_metrics['recall']:.4f}; Official test: acc={official['random_forest']['accuracy']:.4f}, recall={official['random_forest']['recall']:.4f}",
        "Supported?": "Partially",
        "Explanation": "Metrics near 99% reproduce on the same random split setup, but drop sharply on the official NSL-KDD test set.",
    },
    {
        "Claim": "ROC-AUC ≈ 0.9987",
        "Author's Evidence": "Printed ROC-AUC in tutorial output",
        "Our Reproduction Evidence": f"Author-style AUC={author_metrics['roc_auc']:.4f}; Official test AUC={official['random_forest']['roc_auc']:.4f}",
        "Supported?": "Partially",
        "Explanation": "Very high AUC reproduces on the random split; official test AUC remains strong (≈0.97) but not 0.999.",
    },
    {
        "Claim": "Training completes in under 5 minutes",
        "Author's Evidence": "Stated in tutorial text",
        "Our Reproduction Evidence": "Full pipeline completed in a few minutes on a standard laptop CPU",
        "Supported?": "Supported",
        "Explanation": "Runtime claim is realistic for NSL-KDD with sklearn.",
    },
    {
        "Claim": "Isolation Forest helps detect unknown threats",
        "Author's Evidence": "Trains IF on normal traffic and combines with RF in API",
        "Our Reproduction Evidence": f"Official test IF recall={official['isolation_forest']['recall']:.4f} vs RF recall={official['random_forest']['recall']:.4f}",
        "Supported?": "Partially",
        "Explanation": "IF slightly improves attack recall here, but still misses many attacks; not a standalone solution.",
    },
    {
        "Claim": "Real-world performance will be lower than benchmark numbers",
        "Author's Evidence": "Explicit caveat in blog",
        "Our Reproduction Evidence": f"Official test accuracy {official['random_forest']['accuracy']:.4f} vs author-split {author_metrics['accuracy']:.4f}",
        "Supported?": "Supported",
        "Explanation": "Our benchmark vs random-split gap confirms the author's caution.",
    },
])

claims.to_csv(TABLES_DIR / "claim_evaluation.csv", index=False)
display(claims)


## 9. Reproducibility Analysis

| Aspect | Finding |
|--------|---------|
| Original code runs? | Yes — blog code is complete enough to reproduce core training steps |
| Dependencies clear? | Mostly — sklearn, imbalanced-learn, pandas listed |
| Files available? | NSL-KDD must be downloaded separately; not bundled with blog |
| Dataset version clear? | Yes — NSL-KDD train/test files are standard |
| Hidden preprocessing? | No major hidden steps; SMOTE applied after encoding as shown |
| Changes needed? | We additionally evaluate the official `KDDTest+.txt` split for realistic benchmarking |
| Grader reproducibility? | Yes — `pip install -r requirements.txt`, download data script, run notebook |

**Overall judgment:** Moderately reproducible for the tutorial split; production-grade evaluation requires the official test set.


## 10. Experimental Results


In [ ]:
print("Experiments performed:")
print("1. EDA on NSL-KDD train/test")
print("2. Preprocessing + SMOTE (train only)")
print("3. Logistic Regression, Random Forest, Isolation Forest")
print("4. Official test evaluation + author-style split comparison")

print("\nMain metrics table saved to results/tables/model_comparison.csv")
print("Figures saved to results/figures/")

with open(METRICS_DIR / "run_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))


## 11. Conclusions

- The tutorial pipeline is **implementable** and reproduces very high metrics on a random split of the training file.
- On the **official NSL-KDD test set**, attack recall drops to ~0.63–0.67, so high accuracy claims are **misleading for real deployment** if taken at face value.
- **Strengths:** clear preprocessing, SMOTE for imbalance, complementary anomaly model, honest caveat about real-world gap.
- **Weaknesses:** no official test evaluation in the article, no per-attack-type analysis, API severity rules not validated.
- **Recommendation:** use this as an educational baseline, but evaluate on the official test split and prioritize recall/F2 for security operations.


## 12. Executive Summary


In [ ]:
exec_summary = """
This project reproduced a published NSL-KDD intrusion detection tutorial that combines Random Forest,
SMOTE, and Isolation Forest. Using the author's random train/test split, we matched the blog's near-99%
metrics (accuracy 0.9990, ROC-AUC 0.9999). On the official NSL-KDD test set, performance dropped to
accuracy 0.7757 and attack recall 0.6260 for Random Forest, confirming that benchmark claims are highly
split-dependent. The author's warning about real-world degradation is supported. For cybersecurity use,
attack recall and MCC are more informative than accuracy. The approach is useful for learning and prototyping,
but not sufficient alone for production IDS deployment without rigorous official benchmarking and error analysis.
"""
print(exec_summary)

with open(PROJECT_ROOT / "report" / "executive_summary.md", "w", encoding="utf-8") as f:
    f.write("# Executive Summary\n\n" + exec_summary.strip() + "\n")
